# Oxford Pets Project, Simple Notebook Version

This is the simple version of the project workflow. Run the cells from top to bottom.

What this notebook does:

1. Finds the project folder automatically.
2. Checks that the required packages are installed.
3. Loads Oxford Pets with a small student-friendly setting.
4. Shows real dataset images.
5. Builds the scratch CNN and transfer CNN.
6. Runs a tiny training demo.
7. Shows the final result figures used in the report and presentation.

The longer scripts and Makefile are still available, but you do not need them just to understand or demonstrate the project.

## 1. Setup

Run this cell first. If a package is missing, it installs the packages from `requirements.txt` into the Python environment that is currently running this notebook.

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys


def find_project_root() -> Path:
    start = Path.cwd().resolve()
    candidates = [start, *start.parents]
    candidates += [start / "oxford_pets_project", start.parent / "oxford_pets_project"]
    for candidate in candidates:
        if (candidate / "src" / "oxpets").exists() and (candidate / "requirements.txt").exists():
            return candidate.resolve()
    raise RuntimeError("Could not find oxford_pets_project. Open this notebook from inside the project folder.")


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

required_imports = {
    "torch": "torch",
    "torchvision": "torchvision",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "pandas": "pandas",
    "seaborn": "seaborn",
    "tqdm": "tqdm",
    "PIL": "pillow",
}
missing = [package for module, package in required_imports.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements.txt")])

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)
print("Setup ready")

## 2. Load the Dataset

We use a small subset in the notebook so it runs quickly on a laptop. The full project still uses the same code, only with larger limits.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn

from oxpets.config import FIGURES_DIR, METRICS_DIR, TrainConfig, ensure_output_dirs, set_seed
from oxpets.data import TASKS, make_loaders
from oxpets.metrics import compute_metrics, save_confusion_matrix, save_history_plot
from oxpets.models import build_scratch_model, build_transfer_model
from oxpets.train import evaluate, train_model, trainable_parameters

ensure_output_dirs()
set_seed(42)

# CPU is used here because it is the most reliable option for a short notebook demo.
device = torch.device("cpu")
config = TrainConfig(image_size=96, batch_size=8, num_workers=0)

train_loader, val_loader, test_loader, spec = make_loaders(
    task="species",
    config=config,
    download=True,
    limit_train=64,
    limit_val=32,
    limit_test=32,
)

print("Task:", spec.name)
print("Classes:", spec.class_names)
print("Training images in this demo:", len(train_loader.dataset))
print("Validation images in this demo:", len(val_loader.dataset))
print("Test images in this demo:", len(test_loader.dataset))

## 3. Look at Real Pet Images

These are real images from Oxford Pets after resizing and normalization reversal for display.

In [ ]:
images, labels = next(iter(train_loader))

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

plt.figure(figsize=(12, 5))
for index in range(min(8, len(images))):
    image = (images[index] * std + mean).clamp(0, 1).permute(1, 2, 0)
    plt.subplot(2, 4, index + 1)
    plt.imshow(image)
    plt.title(spec.class_names[int(labels[index])])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Build Both Models

The assignment asks for two approaches. Here we check both quickly:

- Scratch CNN: our own convolutional model.
- Transfer CNN: ResNet18 with a replaced classifier head.

In [ ]:
scratch_model = build_scratch_model(spec.num_classes)
transfer_model = build_transfer_model(spec.num_classes, pretrained=False)

with torch.no_grad():
    scratch_logits = scratch_model(images)
    transfer_logits = transfer_model(images)

print("Scratch CNN output shape:", tuple(scratch_logits.shape))
print("Transfer CNN output shape:", tuple(transfer_logits.shape))
print("Each row has one score per class:", spec.class_names)

## 5. Tiny Training Demo

This is not the final accuracy run. It is a short live check to prove that the model, loss, optimizer, and training loop all work.

In [ ]:
model = build_scratch_model(spec.num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(trainable_parameters(model), lr=1e-3, weight_decay=1e-4)

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=2,
)

test_loss, test_accuracy, y_true, y_pred = evaluate(model, test_loader, criterion, device)
metrics = compute_metrics(y_true, y_pred, spec.class_names)

print(f"Demo test accuracy: {test_accuracy:.3f}")
print(f"Demo macro F1: {metrics['macro_f1']:.3f}")

## 6. Save and Show Demo Figures

The report and presentation use generated images like these: training curves, confusion matrices, sample grids, and model comparison plots.

In [ ]:
run_name = "notebook_species_scratch_demo"

history_path = METRICS_DIR / f"{run_name}_history.csv"
metrics_path = METRICS_DIR / f"{run_name}_metrics.json"
history_fig = FIGURES_DIR / f"{run_name}_history.png"
confusion_fig = FIGURES_DIR / f"{run_name}_confusion_matrix.png"

pd.DataFrame(history).to_csv(history_path, index=False)
save_history_plot(history, history_fig, "notebook species scratch demo")
save_confusion_matrix(y_true, y_pred, spec.class_names, confusion_fig, "notebook demo confusion matrix")

for figure_path in [history_fig, confusion_fig]:
    image = plt.imread(figure_path)
    plt.figure(figsize=(8, 5))
    plt.imshow(image)
    plt.axis("off")
    plt.title(figure_path.name)
    plt.show()

## 7. Final Project Figures

These are the already generated result figures from the submitted project. They make the report and presentation more visual while still coming from our own project code.

In [ ]:
final_figures = [
    "dataset_sample_grid.png",
    "augmentation_preview.png",
    "species_transfer_prediction_grid.png",
    "model_comparison_summary.png",
    "species_transfer_history.png",
    "breed_transfer_confusion_matrix.png",
]

for figure_name in final_figures:
    figure_path = FIGURES_DIR / figure_name
    if not figure_path.exists():
        print("Missing figure:", figure_path)
        continue
    image = plt.imread(figure_path)
    plt.figure(figsize=(10, 6))
    plt.imshow(image)
    plt.axis("off")
    plt.title(figure_name)
    plt.show()

## 8. Final Documents

The required exam documents are already prepared here:

- `reports/Project_Report.pdf`
- `slides/Project_Presentation.pdf`

For the normal student workflow, this notebook plus those two PDFs are enough to explain and submit the project. The scripts are included so the results can be reproduced later if needed.